# SIR model of an emerging infection
### Model construction
In this notebook, we will illustrate one of the simplest and most commonly used compartmental models of infectious disease transmission (using our summer modelling package) to represent a new emerging infection "disease X" spreading through an immunologically naive population.

The purpose of this notebook is to show what an SIR model is. This is partly because the SIR framework is one of the commonest modelling frameworks used in infectious disease epidemiological modelling, but also to illustrate what we mean by a "mechanistic" model. The model is mechanistic in that it is an explicit simulation of the population we are interested in, through which we represent the states in which people are in as numbers. There are several advantages to a modelling framework such as this, which we will go into in more detail later in the notebook.

The model consists of:
- Three compartments, named susceptible, infectious and recovered
- A starting population of one million people, one of which is infectious
- An evaluation timespan from time zero to 50 time units (which we can think of as days)
- Inter-compartmental flows for the infection and recovery processes

### Code preamble
The next cell installs our `summer` package imports some standard libraries. You don't need to worry about what these are and can completely ignore this. However, if you happen to have a background in data science, you may be familiar with some of these libraries. We have tried to use some of the commonest libraries that are popular across the field of data science.

In [ ]:
# !pip install git+https://github.com/monash-emu/summer3wip.git@ab4f260aa23d01d1ee50347ea8990c54783ec074

In [ ]:
# Import pandas and numpy for data wrangling and use plotly module for interactive visualisation
import numpy as np
import pandas as pd

pd.options.plotting.backend = "plotly"

# Import various components of our "summer" platform for building models of epidemics
from summer3.graph import defer, Parameter, CompartmentValues
from summer3.epi import (
    Stratification,
    CompartmentMap,
    ManagedArray,
    CategoryData,
    TransitionFlow,
    CompartmentalEpiModel,
)

### Building the model with summer
First, let's create a function that gives us a basic SIR model with its standard compartments, starting populations and flows (transitions) between these compartments implemented. The following diagram shows the structure of the model we are creating:

![](../images/sir_feedback.svg)

Our `summer` library can be used to construct a model with any compartmental structure you like, but we suggest you leave this code unchanged if you're more interested in the epidemiology. The code is intended to be epidemiologically expressive, so don't assume you won't understand it - although we appreciate not everyone likes code! You can easily adjust things like the starting population, the size of the infectious seed and the time period that the simulation will run over at the start of the cell.

We'll include an infection process which is based on frequency-dependent transmission. If you're not familiar with the concept of "frequency-dependent", don't worry. It means that the rate of new infections is proportional to the _prevalence_ of infectious people in the population (rather than the total number of infectious people). What is important to understand is that there is a feedback loop within the system, in that the prevalence of infectious people (the size of the `infectious` compartment) determines the rate at which new people get infected.

N.B. Parameters have to be given a default value, but the value that is used is the one provided when the model's `run` method is called.

In [ ]:
# One million people to run the simulation through
population = 1e6

# One infectious person to introduce into the model
seed = 1.0

# Run from time zero to time 50
start_time = 0
end_time = 50

# Compartment names
model_comps = [
    "susceptible",
    "infectious",
    "recovered",
]

# Nominate the infectious compartment
infect_comps = [
    "infectious",
]

# Build the model object
disease_state = Stratification("disease_state", model_comps)
humans = CompartmentMap.new(disease_state)
sir_model = CompartmentalEpiModel(humans, range(start_time, end_time))

# Set the starting population
start_pop = [
    population - seed,  # Everyone other than the seed susceptible
    seed,  # The infectious seed
    0.0,  # No one recovered at the start
]
sir_model.set_initial_population(
    CategoryData(disease_state.categories(), np.array(start_pop))
)


# Frequency-dependent force of infection
def infection_process(
    comp_values: ManagedArray, contact_rate: float, infectious_comp: tuple
):
    infectious_population = comp_values.query(infectious_comp).sum()
    total_population = comp_values.sum()
    infectious_prevalence = infectious_population / total_population
    return contact_rate * infectious_prevalence


contact_rate = Parameter("effective_contact_rate", 0.0)
force_of_infection = defer(infection_process)(
    CompartmentValues, contact_rate, disease_state["infectious"]
)
infection = TransitionFlow(
    "infection",
    disease_state["susceptible"],
    disease_state["infectious"],
    force_of_infection,
)
sir_model.add_flow(infection)

# Add a recovery transition flow from infectious to recovered
recovery = TransitionFlow(
    "recovery",
    disease_state["infectious"],
    disease_state["recovered"],
    Parameter("recovery_rate", 0.0),
)
sir_model.add_flow(recovery)

### Running the model
Now we have a model set up that we're ready to run. The model is expecting us to give it some parameters, because this is what we promised when we said that the rate of infection and recovery were set with a `Parameter` object. Run this cell and then feel free to adjust the parameters to see how the behaviour of the model changes.

In [ ]:
parameters = {"effective_contact_rate": 1.2, "recovery_rate": 0.2}
results = sir_model.run(parameters)
results["compartments"].to_pandas_df().plot(
    labels={"index": "time", "value": "number of people"}
)

## Interpretation
### Epidemiological quantities
These parameters may not be very intuitive, because they expressed per unit time. So for example, a recovery rate of 0.2 per day implies that if you have a group of infectious people then approximately 20% of them will recover each day. We can also say that the average time infectious is five days.

The contact rate in this model is the number of people who will be infected by an infectious person each day of their infection. Because we have already calculated that they will be infectious for five days, we can calculate the basic reproduction number to be 6.0.

That is:

In [ ]:
infectious_duration = 1.0 / parameters["recovery_rate"]
print(f"The infectious duration is {infectious_duration} days")

basic_reproduction_number = parameters["effective_contact_rate"] * infectious_duration
print(f"The basic reproduction number is {basic_reproduction_number}")

$R_0$ is the (average) number of secondary cases resulting from one case of an infectious disease. In this simple model, because people infect others at a rate given by the `effective_contact_rate` parameter and remain infectious for one `infectious_duration`, the number of people they will infect during their infectious period is just the product of these two quantities.

### Limitations
This is a really simple model that doesn't capture many features of a real infectious disease outbreak. Let's list a few:

- There is no random element to it (i.e. it is deterministic)
- The number of people transitioning between the compartments is calculated by applying rates to compartment sizes, which means compartment values are often not whole numbers
- People become infectious immediately they are infected (i.e. there is no latent period)
- Immunity from infection is complete and permanent
- The rate at which people recover from infection is constant
- There is no heterogeneity between people in terms of their disease outcomes.

Of course, we could build some of these features into the model, depending on how important we thought they were to the infectious disease being simulated and the question we are trying to address. Because this is interactive code, feel free to try!

### Insights
Despite these limitations, even a model as simple as this does capture many of the features that we might expect from a real epidemic of a new infectious disease spreading through a susceptible population. Let's list a few:

- There is an exponential growth phase at the start of the epidemic, the rate of which is determined by $R_0$
- The epidemic turns around and begins to come back down due to the depletion of the susceptible population
- The broad shape of the epidemic looks somewhat like what we might observe in reality